# YOLO26 OBB — Parking Spot Detection Training

Fine-tune a **YOLO26l-obb** model on the parking-spot oriented bounding-box dataset.  
Two classes: `empty_slot` (0) and `occupied_slot` (1).

**Steps**
1. Setup & download dataset from Google Drive
2. Dataset analysis (EDA)
3. Train YOLO26l-obb
4. Evaluate & export to Google Drive

## 1 — Setup & Download Dataset

In [ ]:
!pip install -q ultralytics gdown

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
import gdown
from pathlib import Path

GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/19g9jbKhvHpwJJtYwFYTq8GX8JJEfFrih"
LOCAL_DATASET = Path("/content/dataset")

if not LOCAL_DATASET.exists():
    gdown.download_folder(GDRIVE_FOLDER_URL, output=str(LOCAL_DATASET), quiet=False)
    print(f"Downloaded dataset to {LOCAL_DATASET}")
else:
    print(f"Dataset already at {LOCAL_DATASET}")

!find {LOCAL_DATASET} -type f | head -10

In [ ]:
for split in ["train", "val"]:
    imgs = list((LOCAL_DATASET / split / "images").glob("*.png"))
    lbls = list((LOCAL_DATASET / split / "labels").glob("*.txt"))
    print(f"  {split}: {len(imgs)} images, {len(lbls)} labels")
    assert len(imgs) > 0, f"No images found in {split}/images — check download"
    assert len(imgs) == len(lbls), f"Mismatch: {len(imgs)} images vs {len(lbls)} labels"

### Create `data.yaml`

Ultralytics needs a YAML file describing the dataset layout and class names.

In [ ]:
DATA_YAML = LOCAL_DATASET / "data.yaml"
DATA_YAML.write_text(f"""path: {LOCAL_DATASET}
train: train/images
val: val/images

nc: 2
names:
  0: empty_slot
  1: occupied_slot
""")
print(DATA_YAML.read_text())

## 2 — Dataset Analysis (EDA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import Counter

CLASS_NAMES = {0: "empty_slot", 1: "occupied_slot"}
CLASS_COLORS = {0: "#2ecc71", 1: "#e74c3c"}


def parse_obb_labels(label_dir: Path):
    """Parse all OBB label files. Return list of (class_id, 4-point polygon)."""
    records = []
    for lbl_file in sorted(label_dir.glob("*.txt")):
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            cls_id = int(parts[0])
            coords = list(map(float, parts[1:]))
            polygon = np.array(coords).reshape(4, 2)
            records.append({"file": lbl_file.stem, "class_id": cls_id, "polygon": polygon})
    return records


train_labels = parse_obb_labels(LOCAL_DATASET / "train" / "labels")
val_labels = parse_obb_labels(LOCAL_DATASET / "val" / "labels")
all_labels = train_labels + val_labels

print(f"Total annotations : {len(all_labels)}")
print(f"  Train           : {len(train_labels)}")
print(f"  Val             : {len(val_labels)}")

### 2.1 — Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (title, labels) in zip(axes, [("Train", train_labels), ("Val", val_labels)]):
    counts = Counter(r["class_id"] for r in labels)
    names = [CLASS_NAMES[c] for c in sorted(counts)]
    vals  = [counts[c] for c in sorted(counts)]
    colors = [CLASS_COLORS[c] for c in sorted(counts)]
    bars = ax.bar(names, vals, color=colors, edgecolor="white", linewidth=0.8)
    ax.bar_label(bars, fmt="%d", fontsize=11, fontweight="bold")
    ax.set_title(f"{title} — {sum(vals)} annotations", fontsize=13)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

total_empty    = sum(1 for r in all_labels if r["class_id"] == 0)
total_occupied = sum(1 for r in all_labels if r["class_id"] == 1)
print(f"\nOverall ratio  empty:occupied = {total_empty}:{total_occupied}")
print(f"               empty fraction  = {total_empty / len(all_labels):.1%}")

### 2.2 — Annotations Per Image

In [ ]:
from collections import defaultdict

per_image = defaultdict(int)
for r in all_labels:
    per_image[r["file"]] += 1

counts_per_img = list(per_image.values())

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(
    sorted(per_image.keys()),
    [per_image[k] for k in sorted(per_image.keys())],
    color="#3498db", edgecolor="white", linewidth=0.5,
)
ax.set_xlabel("Number of annotations")
ax.set_title("Annotations per image", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Min: {min(counts_per_img)}  |  Max: {max(counts_per_img)}  |  "
      f"Mean: {np.mean(counts_per_img):.1f}  |  Median: {np.median(counts_per_img):.0f}")

### 2.3 — OBB Size Distribution (width × height in pixels)

In [ ]:
IMG_SIZE = 1280  # all images are 1280×1280

widths, heights = [], []
for r in all_labels:
    poly_px = r["polygon"] * IMG_SIZE
    edge_lengths = np.linalg.norm(np.diff(np.vstack([poly_px, poly_px[0]]), axis=0), axis=1)
    side_a = (edge_lengths[0] + edge_lengths[2]) / 2
    side_b = (edge_lengths[1] + edge_lengths[3]) / 2
    w, h = max(side_a, side_b), min(side_a, side_b)
    widths.append(w)
    heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color="#9b59b6", edgecolor="white")
axes[0].set_title("OBB long-side (px)"); axes[0].set_xlabel("pixels")
axes[1].hist(heights, bins=30, color="#e67e22", edgecolor="white")
axes[1].set_title("OBB short-side (px)"); axes[1].set_xlabel("pixels")
plt.tight_layout(); plt.show()

print(f"Long-side  — mean: {np.mean(widths):.0f}px, median: {np.median(widths):.0f}px")
print(f"Short-side — mean: {np.mean(heights):.0f}px, median: {np.median(heights):.0f}px")

### 2.4 — Sample Visualizations

In [ ]:
def draw_obb_on_image(ax, img_path, labels):
    """Draw OBB polygons on an image."""
    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]
    ax.imshow(img)
    for r in labels:
        poly_px = r["polygon"] * np.array([w, h])
        color = CLASS_COLORS[r["class_id"]]
        polygon = patches.Polygon(poly_px, closed=True, linewidth=1.5,
                                  edgecolor=color, facecolor=color, alpha=0.25)
        ax.add_patch(polygon)
        polygon_edge = patches.Polygon(poly_px, closed=True, linewidth=1.5,
                                       edgecolor=color, facecolor="none")
        ax.add_patch(polygon_edge)
    ax.set_title(img_path.stem, fontsize=10)
    ax.axis("off")


# Group labels by file
labels_by_file = defaultdict(list)
for r in all_labels:
    labels_by_file[r["file"]].append(r)

sample_files = sorted(labels_by_file.keys())[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, stem in zip(axes.flat, sample_files):
    img_path = LOCAL_DATASET / "train" / "images" / f"{stem}.png"
    if not img_path.exists():
        img_path = LOCAL_DATASET / "val" / "images" / f"{stem}.png"
    draw_obb_on_image(ax, img_path, labels_by_file[stem])

fig.legend(
    handles=[
        patches.Patch(facecolor=CLASS_COLORS[0], alpha=0.4, label="empty_slot"),
        patches.Patch(facecolor=CLASS_COLORS[1], alpha=0.4, label="occupied_slot"),
    ],
    loc="upper center", ncol=2, fontsize=12, frameon=False,
)
plt.tight_layout()
plt.show()

## 3 — Train YOLO26l-obb

With only ~22 images we use the **small-dataset recipe** from the Ultralytics docs:
- Frozen backbone (`freeze=10`) to preserve pretrained features
- Reduced augmentation to avoid distorting the few real samples
- Aerial-oriented augmentations (`flipud`, `degrees`) since parking lots are top-down
- Early stopping via `patience`

In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolo26l-obb.pt"

model = YOLO(MODEL_NAME)
print(f"Loaded {MODEL_NAME} — {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f}M params")

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=150,
    patience=30,
    imgsz=1280,
    batch=4,

    # Optimizer — lower LR for small dataset fine-tuning
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,

    # Freeze backbone to avoid overfitting on 22 images
    freeze=10,

    # Augmentation — toned down for small dataset, adapted for aerial/top-down
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
    scale=0.3,
    fliplr=0.5,
    flipud=0.5,       # aerial imagery can be flipped vertically
    degrees=15.0,      # parking spots appear at various angles
    translate=0.1,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,

    close_mosaic=10,
    project="/content/runs",
    name="parking_obb_yolo26l",
    exist_ok=True,
    verbose=True,
)

## 4 — Evaluate

In [ ]:
RUN_DIR = Path("/content/runs/parking_obb_yolo26l")

# Show training curves
results_img = RUN_DIR / "results.png"
if results_img.exists():
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.imshow(Image.open(results_img))
    ax.axis("off")
    ax.set_title("Training curves", fontsize=14)
    plt.show()

In [ ]:
# Show confusion matrix
cm_img = RUN_DIR / "confusion_matrix_normalized.png"
if cm_img.exists():
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(Image.open(cm_img))
    ax.axis("off")
    ax.set_title("Normalized Confusion Matrix", fontsize=14)
    plt.show()

In [ ]:
# Validate best checkpoint on val set
best_weights = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(str(best_weights))

metrics = best_model.val(data=str(DATA_YAML), imgsz=1280)

print(f"\nmAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")
for i, name in CLASS_NAMES.items():
    print(f"  {name:15s}  AP50={metrics.box.ap50[i]:.4f}  AP50-95={metrics.box.ap[i]:.4f}")

### Inference on Validation Images

In [ ]:
val_images = sorted((LOCAL_DATASET / "val" / "images").glob("*.png"))

fig, axes = plt.subplots(1, len(val_images), figsize=(10 * len(val_images), 10))
if len(val_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, val_images):
    preds = best_model.predict(source=str(img_path), imgsz=1280, conf=0.25, verbose=False)
    annotated = preds[0].plot()
    ax.imshow(annotated[..., ::-1])  # BGR → RGB
    ax.set_title(img_path.stem, fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5 — Export & Save to Drive

In [ ]:
# Mount Drive now (only needed for saving results)
from google.colab import drive
drive.mount("/content/drive")

import shutil

DRIVE_OUTPUT = Path("/content/drive/MyDrive/parking_obb_yolo26l_run")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_weights, DRIVE_OUTPUT / "best.pt")
shutil.copy2(RUN_DIR / "weights" / "last.pt", DRIVE_OUTPUT / "last.pt")
shutil.copy2(RUN_DIR / "results.csv", DRIVE_OUTPUT / "results.csv")
if results_img.exists():
    shutil.copy2(results_img, DRIVE_OUTPUT / "results.png")
if cm_img.exists():
    shutil.copy2(cm_img, DRIVE_OUTPUT / "confusion_matrix.png")

print(f"Saved to {DRIVE_OUTPUT}")
!ls -lh "{DRIVE_OUTPUT}"

In [ ]:
# Optional: export to ONNX for deployment
best_model.export(format="onnx", imgsz=1280, simplify=True)
onnx_path = best_weights.with_suffix(".onnx")
if onnx_path.exists():
    shutil.copy2(onnx_path, DRIVE_OUTPUT / "best.onnx")
    print(f"ONNX exported to {DRIVE_OUTPUT / 'best.onnx'}")